# Forecast evaluation
All results derive from stored genuine origin/horizon predictions. Overlapping errors and short holdout windows prevent independent-sample significance claims.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root))
from src.forecast_config import configuration
from src.forecast_models import fit_forecast
from src.forecast_features import supervised_pairs
from src.forecast_metrics import metrics
cfg=configuration(root)
def table(name): return pd.read_parquet(root/f'outputs/phase4/tables/{name}.parquet')
def chart(name): display(Image(filename=str(root/f'outputs/phase4/charts/{name}.png')))
national=pd.read_parquet(root/'outputs/tables/national_monthly_trends.parquet').set_index('month').prescription_items.sort_index()


## Model comparison and baseline
MASE scales each forecast using the mean absolute adjacent training change at its own origin. Bias=prediction−actual. Common-support comparisons remove failed cells for every model. Selected-model summaries retain all selected forecast opportunities.

In [ ]:
display(table('holdout_model_metrics'))
display(table('horizon_metrics').query("split_type == 'holdout' and aggregation_level == 'national'"))
chart('03_national_models')
chart('06_accuracy_horizon')

## Regional errors and residuals
Raw MAE reflects size; origin-scaled MASE complements it. Regions are fixed geographic aggregates, not random independent samples. Errors overlap across horizons.

In [ ]:
display(table('regional_forecast_metrics').query("split_type == 'holdout'"))
chart('08_regional_error_heatmap')
chart('10_residual_distribution')

## Interval support
Only six/five/four development errors per series/horizon are available, below the prespecified minimum of 20. Bounds and empirical coverage are null. A pooled regional residual count cannot supply independent calibration for a national forecast.

In [ ]:
display(table('interval_calibration').query("series_id == 'England'"))
display(table('interval_coverage_summary'))
chart('15_interval_support')

## Hierarchy
National observed items equal all seven regions; each region equals all observed presentations. The screened presentation subset is not the entire population. Direct national and regional-sum forecasts remain separate; coherence gaps are reported without rescaling.

In [ ]:
checks=table('forecast_hierarchy_checks')
assert checks.status.eq('passed').all()
display(table('aggregate_forecast_metrics'))
display(table('aggregate_forecast_comparison').tail())
chart('16_direct_regional_sum')

## Retrospective operational forecasts and limits
Frozen models are refitted through December 2025 to issue January–March 2026 forecasts. These are historical as-of forecasts, not current predictions. No 2026 actuals are fetched or implied. Point forecasts are continuous expected counts; source values are untouched.

In [ ]:
display(table('operational_forecasts').query("series_id == 'England'"))
chart('18_operational_forecasts')
display(table('forecast_robustness_results').head(20))
assert not table('backtest_predictions').duplicated(['series_id','variant','origin_date','horizon','split_type']).any()